# SH-wave modelling — IVF flat free surface (Pan et al. 2018)

Homogeneous half-space with a flat free surface at z = 0.  
The free-surface traction-free condition is enforced via the **Improved Vacuum Formulation** (IVF): three vacuum rows are prepended above z = 0 in the material arrays so that the pre-averaged effective moduli `mu_x` / `mu_z` are zero at the vacuum–solid boundary, automatically driving `tau_zy = 0` there.

Units follow Devito's seismic convention: **km/s**, **kg/m³**, **m**, **ms**.

| Parameter | Value |
|-----------|-------|
| S-wave velocity | 2.02073 km/s |
| Density | 2380.90 kg/m³ |
| Shear modulus μ | ρ × vs² ≈ 9722 kg/m³ · (km/s)² |
| Grid spacing *dx* | 1.25 m |
| Time step *dt* | 0.25 ms |
| Physical domain | 750 × 750 m |
| Vacuum rows (IVF) | 3 rows above z = 0 |
| Source | Ricker 30 Hz, centre of subsurface domain |
| Snapshots | 0.05, 0.10, 0.15, 0.20, 0.25, 0.30 s |

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from devito import TimeFunction, NODE

from examples.seismic import AcquisitionGeometry
from examples.seismic.model import ModelSH
from examples.seismic.sh.operators import ForwardOperator

## 1. Model

In [ ]:
# ── Physical parameters ────────────────────────────────────────────────────────
vs    = 2.02073      # km/s — S-wave velocity
rho   = 2380.90      # kg/m³
mu_val = np.float32(rho * vs**2)   # kg/m³ · (km/s)²  — shear modulus
b_val  = np.float32(1.0 / rho)     # m³/kg — buoyancy

print(f'μ  = {mu_val:.4f}  kg/m³·(km/s)²')
print(f'b  = {b_val:.6e}  m³/kg')

# ── Grid ───────────────────────────────────────────────────────────────────────
dx          = 1.25   # m
dt_ms       = 0.25   # ms  (specified; stability check: critical_dt printed below)
n_vac       = 3      # vacuum rows prepended above z = 0 for IVF
# n_vac       = 0      # vacuum rows prepended above z = 0 for IVF
Nx          = 601    # x nodes  (750 m / 1.25 m = 600 intervals + 1)
Nz_sub      = 601    # z nodes for 750 m subsurface
nbl         = 150     # PML layers on every side
space_order = 6

shape   = (Nx, Nz_sub + n_vac)   # (601, 604)
spacing = (dx, dx)
origin  = (0., -n_vac * dx)       # (0., -3.75) m  — first grid node in z

# ── Build material arrays — caller zeros the vacuum rows (IVF contract) ───────
mu_arr = np.zeros(shape, dtype=np.float32)
b_arr  = np.zeros(shape, dtype=np.float32)
mu_arr[:, n_vac:] = mu_val   # rows 0..2 stay zero → vacuum
b_arr[:, n_vac:]  = b_val

# topo: flat surface at z-index n_vac (z = 0 m)
topo = np.full(Nx, n_vac, dtype=int)

model = ModelSH(
    origin=origin, spacing=spacing, shape=shape,
    space_order=space_order, mu=mu_arr, b=b_arr,
    nbl=nbl, topo=topo, dt=dt_ms,
)

# model = ModelSH(
#     origin=origin, spacing=spacing, shape=shape,
#     space_order=space_order, mu=mu_arr, b=b_arr,
#     nbl=nbl, topo=None, dt=dt_ms,
# )

print(f'\ncritical_dt = {model.critical_dt:.4f} ms  (specified {dt_ms} ms)')
print(f'shape (physical + vacuum) = {shape}')
print(f'padded grid shape         = {model.grid.shape}')
print(f'mu_z at vacuum boundary   = {model.mu_z.data[nbl, nbl + n_vac - 1]:.6f}  (expected 0)')
print(f'mu_z one row below        = {model.mu_z.data[nbl, nbl + n_vac]:.4f}       (expected non-zero)')

## 2. Geometry and time axis

In [ ]:
t0, tn = 0., 300.   # ms
f0 = 0.030          # kHz = 30 Hz

# Source at the centre of the physical subsurface domain (x=375 m, z=375 m)
src_x = (Nx - 1) * dx / 2.       # 375.0 m
src_z = (Nz_sub - 1) * dx / 2.   # 375.0 m

src_positions = np.array([[src_x, src_z]], dtype=np.float32)

# Receiver line along the free surface (z = 0)
rec_x = np.linspace(dx, (Nx - 2) * dx, 300)
rec_positions = np.column_stack([rec_x, np.zeros_like(rec_x)]).astype(np.float32)

geometry = AcquisitionGeometry(
    model, rec_positions, src_positions,
    t0=t0, tn=tn, src_type='Ricker', f0=f0,
)

dt = model.critical_dt   # = 0.25 ms as specified
print(f'Source : ({src_x:.1f} m, {src_z:.1f} m)')
print(f'dt     : {dt} ms,  nt = {geometry.nt}')

## 3. Operator and wavefield buffers

In [ ]:
op = ForwardOperator(model, geometry, space_order=space_order)

x, z = model.grid.dimensions

v = TimeFunction(name='v', grid=model.grid, space_order=space_order,
                 time_order=1, staggered=NODE)
tau_xy = TimeFunction(name='tau_xy', grid=model.grid, space_order=space_order,
                      time_order=1, staggered=(x,))
tau_zy = TimeFunction(name='tau_zy', grid=model.grid, space_order=space_order,
                      time_order=1, staggered=(z,))
rec = geometry.new_rec(name='rec')

## 4. Run in segments — capture wavefield at each snapshot time

In [ ]:
snap_times_ms = [50., 100., 150., 200., 250., 300.]
snap_steps    = [round(t / dt) for t in snap_times_ms]

print('Snapshot steps:', snap_steps)

snapshots = []
prev = 0

for step in snap_steps:
    op.apply(
        v=v, tau_xy=tau_xy, tau_zy=tau_zy,
        src=geometry.src, rec=rec,
        time_m=prev, time_M=step,
        dt=dt, **model.physical_params(),
    )
    # time_order=1 ring buffer: slot of the most recent field after time_M steps
    slot = (step + 1) % 2
    data = v.data[slot, nbl:-nbl, nbl:-nbl].copy()   # shape (601, 604)
    snapshots.append(data)
    prev = step + 1
    print(f't = {step * dt:6.1f} ms   |v|_max = {np.abs(data).max():.4e}')

## 5. Snapshots

Each panel shows the particle velocity `v` at the indicated time.  
The dashed line marks the IVF free surface at z = 0; the star marks the source.  
Colour is normalised independently per frame so the wavefront remains visible throughout.

In [ ]:
# Physical-domain extent [m]
x_min, x_max = origin[0], origin[0] + (Nx - 1) * dx          # 0 … 750 m
z_min, z_max = origin[1], origin[1] + (Nz_sub + n_vac - 1) * dx  # -3.75 … 750 m

# imshow extent=(left, right, bottom, top)  — z increases downward
extent = [x_min, x_max, z_max, z_min]

fig, axes = plt.subplots(2, 3, figsize=(15, 10), constrained_layout=True)

for ax, data, t_ms in zip(axes.flat, snapshots, snap_times_ms):
    vmax = np.abs(data).max() or 1.0
    im = ax.imshow(
        data.T,
        aspect='equal',
        extent=extent,
        cmap='RdBu',
        vmin=-vmax, vmax=vmax,
        interpolation='nearest',
    )
    ax.axhline(0., color='k', linewidth=1.2, linestyle='--', label='free surface (z=0)')
    ax.plot(src_x, src_z, 'k*', markersize=9, label='source')
    ax.set_title(f't = {t_ms / 1000:.2f} s', fontsize=12)
    ax.set_xlabel('x [m]')
    ax.set_ylabel('z [m]')
    # plt.colorbar(im, ax=ax, shrink=0.75, label='v [m/s]')

# axes.flat[0].legend(fontsize=8, loc='lower right')
fig.suptitle('SH wavefield snapshots — IVF free surface (Pan et al. 2018)', fontsize=14)
plt.show()